# 🛒 Walmart Store Sales Prediction — Linear Regression

**Автор:** [Your Name]  
**Дата:** Июнь 2026  
**Датасет:** Walmart Store Sales (6,435 строк × 8 признаков)  
**Задача:** Предсказать недельные продажи на основе макроэкономических и операционных факторов.

---

## 📋 Problem Statement

Walmart — крупнейший ритейлер мира, управляющий тысячами магазинов в разных регионах.  
**Вопрос:** *Какие внешние факторы — праздники, уровень безработицы, инфляция, цены на топливо, температура — определяют недельные продажи в магазине?*

Эта задача — классический **supervised regression**: нужно предсказать непрерывную величину (Weekly Sales) на основе численных признаков.

**Почему Linear Regression?**  
- Прозрачность: коэффициенты напрямую интерпретируются как "вклад признака"  
- Базовая модель: сначала объяснимая простая модель, потом усложнять  
- Академически обоснована для экономических данных

---

## 📦 Dataset Description

| Признак | Тип | Описание |
|---|---|---|
| `Store` | int | Номер магазина (1–45) |
| `Date` | date | Дата недели |
| `Weekly_Sales` | float | **Целевая переменная** — недельная выручка магазина ($) |
| `Holiday_Flag` | bool | 1 если праздничная неделя (Super Bowl, Thanksgiving, Christmas, Labor Day) |
| `Temperature` | float | Средняя температура в регионе (°F) |
| `Fuel_Price` | float | Цена топлива в регионе ($/gallon) |
| `CPI` | float | Consumer Price Index — индекс потребительских цен |
| `Unemployment` | float | Уровень безработицы в регионе (%) |

**Период:** февраль 2010 — октябрь 2012  
**Источник:** Kaggle — Walmart Store Sales Dataset


In [ ]:
# ── Импорты ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 13
sns.set_theme(style='whitegrid', palette='muted')

print("✅ Библиотеки загружены")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

## 1. Загрузка и первичный осмотр данных

In [ ]:
df = pd.read_csv('../data/Walmart_Store_Sales.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week']  = df['Date'].dt.isocalendar().week.astype(int)

print(f"Форма датасета: {df.shape[0]} строк × {df.shape[1]} колонок")
print(f"\nПервые 5 строк:")
df.head()

## 2. Exploratory Data Analysis (EDA)

### 2.1 Базовая статистика

In [ ]:
# Описательная статистика
print("🔎 Описательная статистика числовых признаков:\n")
print(df[['Weekly_Sales','Temperature','Fuel_Price','CPI','Unemployment']].describe().round(3))

print(f"\n📌 Пропущенные значения:\n{df.isnull().sum()}")
print(f"\n📌 Дубликатов: {df.duplicated().sum()}")
print(f"\n📌 Уникальных магазинов: {df['Store'].nunique()}")
print(f"📌 Временной период: {df['Date'].min().date()} → {df['Date'].max().date()}")

### 2.2 Распределение признаков

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Распределение признаков — Walmart Sales Dataset',
             fontsize=15, fontweight='bold', y=1.01)

features_hist = [
    ('Weekly_Sales', '#1E88E5', 'Weekly Sales ($)'),
    ('Temperature',  '#43A047', 'Temperature (°F)'),
    ('Fuel_Price',   '#FB8C00', 'Fuel Price ($/gal)'),
    ('CPI',          '#8E24AA', 'CPI (Consumer Price Index)'),
    ('Unemployment', '#E53935', 'Unemployment Rate (%)'),
    ('Holiday_Flag', '#00ACC1', 'Holiday Flag'),
]

for ax, (feat, col, label) in zip(axes.flatten(), features_hist):
    if feat == 'Holiday_Flag':
        counts = df[feat].value_counts()
        ax.bar(['Non-Holiday (0)', 'Holiday (1)'], counts.values,
               color=[col, '#B0BEC5'], edgecolor='white', alpha=0.9)
        ax.set_title('Holiday Flag\n(1 = праздничная неделя)', fontweight='bold')
    else:
        ax.hist(df[feat], bins=40, color=col, edgecolor='white', alpha=0.85)
        ax.set_title(label, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../images/1_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Комментарий
print("""
💬 Наблюдения:
• Weekly_Sales имеет правостороннее распределение (right-skewed) — 
  большинство магазинов продают ~$1–2M в неделю, но есть крупные магазины с $3–4M.
• Temperature равномерно распределена по всем сезонам — данные покрывают 2.5 года.
• Fuel_Price плавно растёт — инфляция топлива 2010–2012.
• Unemployment сосредоточен в диапазоне 7–9% — период после кризиса 2008.
• Holiday_Flag сильно несбалансирован: праздничные недели редки (~5%), 
  что реалистично — в году ~4 праздничных периода.
""")

### 2.3 Нормализация: убираем store-level эффект

> **Проблема:** магазин типа A (площадь 200k sq ft) продаёт в 5–10 раз больше, 
> чем магазин типа C (34k sq ft). Если обучать модель на сырых данных, 
> линейная регрессия будет "видеть" в основном разницу между магазинами, 
> а не влияние внешних факторов (температуры, безработицы и т.д.).
>
> **Решение:** Z-score нормализация внутри каждого магазина. 
> Теперь модель предсказывает *отклонение от нормы* конкретного магазина.

In [ ]:
# Z-score нормализация внутри каждого Store
df['Sales_z'] = df.groupby('Store')['Weekly_Sales'].transform(
    lambda x: (x - x.mean()) / x.std()
)

print("Проверка нормализации:")
print(f"  Sales_z mean: {df['Sales_z'].mean():.4f}  (должно быть ≈0)")
print(f"  Sales_z std:  {df['Sales_z'].std():.4f}   (должно быть ≈1)")
print(f"\nТеперь Sales_z=+1.0 означает: продажи на 1σ выше нормы для этого магазина")
print(f"Sales_z=-1.0 означает: продажи на 1σ ниже нормы для этого магазина")

### 2.4 Корреляционная матрица

In [ ]:
num_cols = ['Sales_z','Temperature','Fuel_Price','CPI','Unemployment','Holiday_Flag','Month','Week']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 11, 'fontweight': 'bold'})
ax.set_title('Корреляционная матрица — признаки vs Sales_z',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../images/2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Корреляция признаков с Sales_z:")
print(corr['Sales_z'].drop('Sales_z').sort_values(ascending=False).round(4).to_string())

print("""
💬 Ключевые наблюдения:
• Temperature (r=+0.59) — сильнейшая корреляция: летом продажи выше.
  Объяснение: BBQ сезон, школьные покупки перед сентябрём.
• Holiday_Flag (r=+0.21) — праздничные недели дают заметный буст.
• Month и Week (r≈-0.33) — сезонный эффект: январь-февраль (начало года) — спад.
• Fuel_Price и CPI (r≈-0.13) — рост цен немного давит на реальные продажи.
• Unemployment (r≈+0.11) — небольшая неожиданная положительная корреляция 
  (объяснение: период экономического восстановления 2010–2012).
""")

### 2.5 Scatter: каждый признак vs Sales_z

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Scatter plots: признаки vs нормированные продажи (Sales_z)',
             fontsize=14, fontweight='bold')

scatter_feats = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
colors_scatter = ['#1565C0', '#2E7D32', '#E65100', '#B71C1C']

for ax, feat, col in zip(axes.flatten(), scatter_feats, colors_scatter):
    x = df[feat]
    y = df['Sales_z']
    ax.scatter(x, y, alpha=0.12, s=12, color=col)
    # Линия тренда
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x.min(), x.max(), 300)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2.5, label='тренд')
    r = df[[feat, 'Sales_z']].corr().iloc[0, 1]
    ax.set_xlabel(feat, fontweight='bold')
    ax.set_ylabel('Sales_z', fontweight='bold')
    ax.set_title(f'{feat}  |  Pearson r = {r:.3f}', fontweight='bold')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../images/3_scatter_features_vs_sales.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.6 Сезонный анализ и праздники

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Сезонный анализ продаж', fontsize=14, fontweight='bold')

# Boxplot праздник vs нет
groups = [df.loc[df['Holiday_Flag'] == 0, 'Weekly_Sales'].values / 1e6,
          df.loc[df['Holiday_Flag'] == 1, 'Weekly_Sales'].values / 1e6]
bp = axes[0].boxplot(groups, patch_artist=True, widths=0.5,
                     boxprops=dict(facecolor='#90CAF9', color='#1565C0'),
                     medianprops=dict(color='#E53935', linewidth=2.5),
                     whiskerprops=dict(color='#1565C0'),
                     capprops=dict(color='#1565C0'))
axes[0].set_xticklabels(['Non-Holiday\n(обычная неделя)', 'Holiday\n(праздник)'])
axes[0].set_title('Распределение продаж\nпраздник vs обычная неделя', fontweight='bold')
axes[0].set_ylabel('Weekly Sales ($M)')

# Месячные средние
monthly = df.groupby('Month')['Weekly_Sales'].mean() / 1e6
month_names = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']
bar_colors = ['#E53935' if m == 12 else '#42A5F5' for m in range(1, 13)]
axes[1].bar(range(1, 13), monthly.values, color=bar_colors, edgecolor='white', alpha=0.9)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names, fontsize=10)
axes[1].set_title('Средние продажи по месяцам\n(Декабрь = красный — пик)', fontweight='bold')
axes[1].set_ylabel('Avg Weekly Sales ($M)')

plt.tight_layout()
plt.savefig('../images/4_seasonal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

h_avg = df.groupby('Holiday_Flag')['Weekly_Sales'].mean()
print(f"💡 Holiday uplift: +{(h_avg[1]/h_avg[0]-1)*100:.1f}% к продажам в праздничные недели")
print(f"   Non-Holiday avg: ${h_avg[0]:,.0f}")
print(f"   Holiday avg:     ${h_avg[1]:,.0f}")

## 3. Обучение модели — Multiple Linear Regression

### 3.1 Подготовка данных

In [ ]:
feature_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
                'Holiday_Flag', 'Month', 'Week']
target_col   = 'Sales_z'

X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Признаки:    {feature_cols}")
print(f"✅ Цель:        {target_col}")
print(f"✅ Train:       {X_train.shape[0]} строк ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"✅ Test:        {X_test.shape[0]} строк ({X_test.shape[0]/len(X)*100:.0f}%)")

### 3.2 Обучение и предсказание

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

print("✅ Модель обучена!")
print(f"\nУравнение модели:")
print(f"  Sales_z = {model.intercept_:.4f}", end='')
for feat, coef in zip(feature_cols, model.coef_):
    sign = '+' if coef >= 0 else '-'
    print(f" {sign} {abs(coef):.5f}×{feat}", end='')
print()

## 4. Оценка модели

### 4.1 MAE, RMSE, R²

> **Почему мы выбрали R² как основную метрику для сравнения?**  
> Потому что MAE и RMSE не сравнимы напрямую между собой (разные единицы интерпретации).  
> R² — универсальная метрика [0, 1]: 0 = модель не лучше среднего, 1 = идеально.

In [ ]:
def compute_metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"\n── {label} ──────────────────────────────")
    print(f"  MAE:   {mae:.4f}  ← среднее абс. отклонение (в σ продаж)")
    print(f"  RMSE:  {rmse:.4f}  ← ср.-кв. отклонение (штрафует крупные ошибки)")
    print(f"  R²:    {r2:.4f}  ← модель объясняет {r2*100:.1f}% дисперсии")
    return mae, rmse, r2

mae_tr, rmse_tr, r2_tr = compute_metrics(y_train, y_pred_train, "TRAIN SET")
mae_te, rmse_te, r2_te = compute_metrics(y_test,  y_pred_test,  "TEST SET")

# Cross-validation
cv_r2 = cross_val_score(model, X, y, cv=10, scoring='r2')
print(f"\n── 10-Fold Cross-Validation ──────────────────────")
print(f"  R² по каждому fold: {[f'{v:.3f}' for v in cv_r2]}")
print(f"  Mean R²: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

print(f"""
💬 Интерпретация:
• R² ≈ 0.50: модель объясняет ~50% вариации в продажах магазина.
  Для линейной модели с 7 простыми признаками — это хороший результат.
• Train R² ≈ Test R²: нет overfitting — модель обобщается.
• CV std = {cv_r2.std():.4f}: стабильно работает на разных подвыборках.
• MAE ≈ 0.57σ: ошибка примерно 0.57 стандартного отклонения продаж магазина.
""")

### 4.2 Actual vs Predicted + Residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Качество модели: Actual vs Predicted', fontsize=14, fontweight='bold')

# Scatter
ax = axes[0]
ax.scatter(y_test, y_pred_test, alpha=0.3, s=15, color='#1565C0', label='Предсказания')
mn, mx = min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=2.5, label='Идеальная линия (y=x)')
ax.set_xlabel('Actual Sales_z', fontweight='bold')
ax.set_ylabel('Predicted Sales_z', fontweight='bold')
ax.set_title(f'Actual vs Predicted  |  R² = {r2_te:.3f}', fontweight='bold')
ax.legend()

# Residuals
residuals = y_test - y_pred_test
ax = axes[1]
ax.hist(residuals, bins=60, color='#EF5350', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', lw=2, linestyle='--', label='Нулевая ошибка')
ax.set_xlabel('Residuals (Sales_z)', fontweight='bold')
ax.set_ylabel('Количество', fontweight='bold')
ax.set_title(f'Распределение ошибок (Residuals)\nmean={residuals.mean():.4f}, std={residuals.std():.4f}',
             fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../images/5_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💬 Интерпретация:
• Точки группируются вдоль красной диагонали — модель в целом работает.
• Residuals почти нормально распределены с центром в 0 — нет систематической ошибки.
• Остаётся значительный разброс: часть дисперсии продаж — случайная или зависит
  от признаков, которых у нас нет (акции, ассортимент, конкуренты рядом).
""")

## 5. Интерпретация коэффициентов

> Это самая важная часть в линейной регрессии — понять **что и как** влияет.
> Используем стандартизованные коэффициенты, чтобы признаки были сравнимы между собой.

In [ ]:
# Стандартизованные коэффициенты
scaler_std = StandardScaler()
X_scaled   = scaler_std.fit_transform(X)
model_std  = LinearRegression().fit(X_scaled, y)

coef_df = pd.DataFrame({
    'Feature':    feature_cols,
    'Raw_Coef':   model.coef_,
    'Std_Coef':   model_std.coef_
}).sort_values('Std_Coef', key=abs, ascending=True)

print("Стандартизованные коэффициенты (сравнимые между собой):")
print(coef_df[['Feature','Raw_Coef','Std_Coef']].to_string(index=False))

print("""
📖 Интерпретация каждого коэффициента:

1. Temperature (+0.856):  СИЛЬНЕЙШИЙ признак. 
   +1σ температуры → продажи на 0.856σ выше нормы магазина.
   Почему? Тёплое лето → BBQ, садовые товары, обратно в школу.

2. Holiday_Flag (+0.302): Праздник → продажи на 0.3σ выше нормы.
   Super Bowl, Thanksgiving, Christmas — американский ритуал шопинга.

3. Week (+0.206): Поздние недели года (осень-зима) → рост.
   Отражает предрождественский shopping season.

4. Month (+0.087): Аналогично Week — позднее в году, выше продажи.

5. Fuel_Price (-0.027): Дорогое топливо → люди реже едут в магазин.
   Малый эффект: Walmart-покупатели часто едут всё равно.

6. CPI (-0.003): Инфляция незначительно давит на реальные продажи.

7. Unemployment (-0.002): Рост безработицы → меньше денег у населения.
   Малый эффект здесь из-за временного периода восстановления.
""")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#1E88E5' if c > 0 else '#E53935' for c in coef_df['Std_Coef']]
bars = ax.barh(coef_df['Feature'], coef_df['Std_Coef'],
               color=bar_colors, edgecolor='white', alpha=0.9, height=0.6)
ax.axvline(0, color='black', linewidth=1.5)
ax.set_xlabel('Стандартизованный коэффициент\n(влияние на 1σ изменения в продажах)',
              fontweight='bold')
ax.set_title('Важность признаков — Linear Regression\n(синий = рост продаж | красный = снижение)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, coef_df['Std_Coef']):
    ax.text(val + (0.01 if val >= 0 else -0.01),
            bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/6_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Сводный график метрик

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Итоговые метрики модели (Train vs Test)', fontsize=14, fontweight='bold')

metric_info = [
    ('MAE',  [mae_tr, mae_te],   'Меньше = лучше\n(ср. абс. ошибка в σ)'),
    ('RMSE', [rmse_tr, rmse_te], 'Меньше = лучше\n(штраф за большие ошибки)'),
    ('R²',   [r2_tr, r2_te],     'Больше = лучше\n(0=случайно, 1=идеально)'),
]
for ax, (name, vals, note) in zip(axes, metric_info):
    bars = ax.bar(['Train', 'Test'], vals,
                  color=['#42A5F5','#EF5350'], edgecolor='white', width=0.5, alpha=0.9)
    ax.set_title(f'{name}\n{note}', fontweight='bold', fontsize=11)
    ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=12, fontweight='bold')
    if name == 'R²':
        ax.set_ylim(0, 0.7)

plt.tight_layout()
plt.savefig('../images/7_metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Выводы и ограничения модели

### ✅ Что получилось

| Метрика | Train | Test | CV (10-fold) |
|---|---|---|---|
| MAE | 0.560 σ | 0.575 σ | — |
| RMSE | 0.721 σ | 0.724 σ | — |
| R² | 0.471 | **0.495** | 0.475 ± 0.021 |

Модель объясняет ~50% дисперсии в нормированных продажах.  
Стабильность между Train/Test и на 10-fold CV говорит об отсутствии overfitting.

### 🔑 Главные инсайты
- **Temperature** — самый сильный предиктор: летний шопинг сезон значимо поднимает продажи  
- **Holiday weeks** дают +11.7% буст — праздничные традиции Америки работают  
- **Безработица и инфляция** — ожидаемо давят на потребление, но эффект умеренный

### ⚠️ Ограничения модели
1. **Линейность**: модель предполагает линейные зависимости, тогда как в ритейле много нелинейных взаимодействий (например, Holiday × Temperature)
2. **Нет store-level fixed effects**: z-score нормализация частично решает проблему, но не полностью
3. **Нет данных по отделам**: разные отделы (groceries vs electronics) ведут себя по-разному
4. **Временной период**: данные 2010–2012, поведение покупателей существенно изменилось
5. **Нет данных о конкурентах**: наличие Target или Costco рядом влияет на продажи

### 🚀 Что можно улучшить
- Добавить interaction features: `Holiday × Month`, `Temperature²`
- Попробовать Ridge/Lasso регрессию для регуляризации
- Добавить store-level признаки (размер, тип магазина)
- Рассмотреть Random Forest как сравнение с baseline
